# Read cube.ply as text rows

`public/cube.ply` is a **binary_little_endian** PLY. The header is ASCII text; the body after `end_header` is raw binary. Reading every line as text shows the header cleanly and renders binary bytes as replacement chars.

In [ ]:
from pathlib import Path

PLY_PATH = Path("..") / "public" / "cube.ply"
print(PLY_PATH.resolve())
print("exists:", PLY_PATH.exists())

In [ ]:
# Read whole file as text rows (binary bytes -> replacement char)
rows = PLY_PATH.read_text(encoding="utf-8", errors="replace").splitlines()

for i, row in enumerate(rows):
    print(f"{i:>3}: {row!r}")

In [ ]:
# Header rows only (up to and including end_header)
header = []
for row in rows:
    header.append(row)
    if row.strip() == "end_header":
        break

for i, row in enumerate(header):
    print(f"{i:>3}: {row}")

## All rows as readable text

Decode the binary body into readable values. Element counts and vertex properties are parsed from the header (no hard-coded sizes), then every vertex and face row is printed as text.

In [ ]:
import struct

# struct char per PLY scalar type
PLY_FMT = {
    "char": "b", "uchar": "B", "int8": "b", "uint8": "B",
    "short": "h", "ushort": "H", "int16": "h", "uint16": "H",
    "int": "i", "uint": "I", "int32": "i", "uint32": "I",
    "float": "f", "float32": "f", "double": "d", "float64": "d",
}

data = PLY_PATH.read_bytes()
marker = b"end_header\n"
body = data[data.index(marker) + len(marker):]

# Parse header -> list of (element_name, count, [(prop_name, type_or_list), ...])
elements = []
for line in header:
    parts = line.split()
    if not parts:
        continue
    if parts[0] == "element":
        elements.append((parts[1], int(parts[2]), []))
    elif parts[0] == "property" and elements:
        if parts[1] == "list":
            # property list <count_type> <item_type> <name>
            elements[-1][2].append((parts[4], ("list", parts[2], parts[3])))
        else:
            elements[-1][2].append((parts[2], parts[1]))

# Walk the body, decoding each element's rows
off = 0
for name, count, props in elements:
    print(f"=== {name} ({count}) ===")
    for i in range(count):
        vals = []
        for pname, ptype in props:
            if isinstance(ptype, tuple):  # list property
                _, count_type, item_type = ptype
                (n,) = struct.unpack_from("<" + PLY_FMT[count_type], body, off)
                off += struct.calcsize(PLY_FMT[count_type])
                fmt = "<" + PLY_FMT[item_type] * n
                items = list(struct.unpack_from(fmt, body, off))
                off += struct.calcsize(fmt)
                vals.append(f"{pname}={items}")
            else:
                fmt = "<" + PLY_FMT[ptype]
                (v,) = struct.unpack_from(fmt, body, off)
                off += struct.calcsize(fmt)
                vals.append(f"{pname}={v:g}" if isinstance(v, float) else f"{pname}={v}")
        print(f"  {name}[{i}]: " + "  ".join(vals))

print(f"\ndecoded {off} of {len(body)} body bytes")